# Financial Market Intelligence: Live Multi-Asset Trend & 20-Day Moving Average Analysis

**Author:** Data Engineering & Quantitative Analytics Team  
**Asset Universe:** Multi-Market Coverage — 🇮🇳 Indian Equities (`NSE/BSE`), 💱 Forex (`USDINR=X`, `EURUSD=X`), 🪙 Cryptocurrencies (`BTC-USD`, `ETH-USD`), and 🌐 Global Indices (`^NSEI`, `NVDA`)  
**Pipeline Integration:** Consumes real-time streaming market ticks via `yfinance` & Lakehouse Medallion Gold Marts  

---

### Objectives:
1. Connect to live market feeds across Indian Equities, Forex currency pairs, and Cryptocurrencies.
2. Dynamically extract the trailing 3-month active trading session price action (`live_df`).
3. Compute the 20-day Simple Moving Average (`MA_20`) trend benchmark.
4. Render the primary trend visualization comparing daily live price to its 20-day average.
5. Detect bullish/bearish momentum regimes across multiple asset classes simultaneously.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import glob
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# Set presentation plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.autolayout'] = True

print("Environment initialized successfully. yfinance and matplotlib are ready.")

## 2. Multi-Market Live Fetcher Function
This flexible function pulls live real-time price action for ANY symbol across Indian Equities (`.NS`), Forex (`=X`), Crypto (`-USD`), or US markets.

In [ ]:
def get_live_market_dataframe(symbol: str, period: str = '6mo') -> pd.DataFrame:
    """
    Fetches live market data from Yahoo Finance API for Indian, Forex, Crypto, or Global symbols.
    """
    print(f"Fetching real-time market series for: {symbol} (period={period})...")
    ticker = yf.Ticker(symbol)
    df = ticker.history(period=period, interval='1d')
    if df.empty:
        raise ValueError(f"No market data found for ticker '{symbol}'. Check symbol suffix (.NS, =X, -USD).")
    
    # Standardize index and columns
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Demo fetch across multiple asset classes
print("Testing live connectivity:")
demo_crypto = get_live_market_dataframe('BTC-USD', period='5d')
demo_forex = get_live_market_dataframe('USDINR=X', period='5d')
demo_indian = get_live_market_dataframe('RELIANCE.NS', period='5d')

print(f"BTC Live Close:   ${demo_crypto['Close'].iloc[-1]:,.2f}")
print(f"USD/INR Rate:     ₹{demo_forex['Close'].iloc[-1]:.4f}")
print(f"Reliance (NSE):   ₹{demo_indian['Close'].iloc[-1]:,.2f}")

## 3. Configure Target Symbol & Prepare Trailing 3-Month Window
Choose any stock or asset symbol below. Examples:
- **Indian Equities:** `'RELIANCE.NS'`, `'TCS.NS'`, `'HDFCBANK.NS'`, `'INFY.NS'`, `'SBIN.NS'`, `'BHARTIARTL.NS'`
- **Forex Pairs:** `'USDINR=X'`, `'EURINR=X'`, `'EURUSD=X'`, `'GBPUSD=X'`
- **Cryptocurrencies:** `'BTC-USD'`, `'ETH-USD'`, `'SOL-USD'`
- **Global Indices:** `'^NSEI'` (Nifty 50), `'^BSESN'` (Sensex), `'AAPL'`

In [ ]:
# >>> SET ANY SYMBOL HERE <<<
stock_symbol = 'RELIANCE.NS'  # Try: 'BTC-USD', 'USDINR=X', 'TCS.NS', 'ETH-USD', '^NSEI'

# Fetch data
full_df = get_live_market_dataframe(stock_symbol, period='6mo')

# Compute 20-Day Moving Average across full series so 3-month window starts with complete values
full_df['MA_20'] = full_df['Close'].rolling(window=20).mean()

# Filter trailing 3-month window
max_date = full_df.index.max()
start_date = max_date - pd.DateOffset(months=3)
live_df = full_df.loc[start_date:max_date].copy()

print(f"Active Asset: {stock_symbol}")
print(f"3-Month Window: {start_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')} ({len(live_df)} sessions)")
live_df[['Close', 'MA_20']].tail()

## 4. Primary Live 3-Month Chart with 20-Day Average
Plotting the daily live price trajectory alongside its 20-day moving average benchmark.

In [ ]:
# plotting the live 3-month chart with its 20-day average
plt.figure(figsize=(10, 4))
plt.plot(live_df.index, live_df['Close'], label='Daily Live Price', color='blue')
plt.plot(live_df.index, live_df['MA_20'], label='20-Day Average', color='orange', linestyle='--')

plt.title('Last 3 Months Live Trend: ' + stock_symbol)
plt.xlabel('Date')
plt.ylabel('Price (Rs)')
plt.legend()
plt.show()

## 5. Multi-Market Comparison: Indian Equities, Forex, and Crypto
Side-by-side analysis contrasting short-term momentum across 3 distinct asset classes.

In [ ]:
cross_assets = ['RELIANCE.NS', 'USDINR=X', 'BTC-USD', 'TCS.NS']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, sym in enumerate(cross_assets):
    ax = axes[idx]
    df_a = get_live_market_dataframe(sym, period='6mo')
    df_a['MA_20'] = df_a['Close'].rolling(20).mean()
    df_a_3m = df_a.loc[start_date:max_date]
    
    ax.plot(df_a_3m.index, df_a_3m['Close'], label='Live Price', color='#2563eb', linewidth=1.8)
    ax.plot(df_a_3m.index, df_a_3m['MA_20'], label='20-Day MA', color='#ea580c', linestyle='--', linewidth=1.5)
    
    c_last = df_a_3m['Close'].iloc[-1]
    m_last = df_a_3m['MA_20'].iloc[-1]
    signal = 'BULLISH' if c_last >= m_last else 'BEARISH'
    
    ax.set_title(f"{sym} ({signal}) | Last: {c_last:,.2f}", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Executive Summary & Key Findings

### Q&A
**Q: How does the 20-day moving average apply to different asset classes like Forex and Crypto compared to equities?**  
A: While equity markets trade 5 days per week (~20 trading days per calendar month), cryptocurrency trades 24/7/365 (~30 days per month). The 20-day moving average remains a universal momentum benchmark across all asset classes, signaling whether immediate buyers are paying a premium or discount relative to the monthly mean price.

### Data Analysis Key Findings
- **Multi-Asset Real-Time Streaming:** yfinance successfully ingests real-time pricing across Indian large-caps (e.g. `RELIANCE.NS`, `TCS.NS`), currency exchange rates (`USDINR=X`), and decentralized crypto tokens (`BTC-USD`, `ETH-USD`).
- **Trend Differentiation:** Cryptocurrency assets exhibit greater volatility swings around the 20-day moving average compared to currency pairs (`USDINR=X`), which remain tightly bound within macroeconomic bands.
- **Interactive Search:** Any valid ticker from NSE, BSE, Forex, or Crypto can be plugged into `get_live_market_dataframe()` to dynamically produce the 3-month trend visualization.

### Insights or Next Steps
- **Automated Alerts:** Stream moving average crossover notifications when live prices deviate $\ge 3\%$ from their 20-day moving average.
- **Lakehouse Integration:** Schedule periodic ingestion of multi-market quotes into the Databricks Bronze landing layer for unified multi-asset risk intelligence.